In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import shap
from sklearn.preprocessing import StandardScaler

sys.path.append("..")
from modeling import COVARIATES, build_lgbm, preprocess_fold

DATA_DIR = Path("/home/ec2-user/studies/Haochen_food/code_micro/data")
TRAIN_DIR = DATA_DIR / "train"
OUT_DIR = TRAIN_DIR / "shap"

model_input = pd.read_csv(TRAIN_DIR / "model_input_diet_cov_oral_species_abundance_baseline.csv", low_memory=False)
summary = pd.read_csv(TRAIN_DIR / "model_species_abundance_summary_baseline.csv")
permutation = pd.read_csv(TRAIN_DIR / "permutation_tables" / "permutation_results_species_abundance_baseline.csv")
targets = permutation.loc[permutation["perm_significant_fdr"], "species"].tolist()
features = COVARIATES + [c for c in model_input.columns if c not in ["participant_id", *COVARIATES, *summary["species"]]]
x, _ = preprocess_fold(model_input[features], model_input[features])

In [ ]:
mean_abs_shap = {}
for target in targets:
    y = model_input[target].to_numpy(dtype=float).reshape(-1, 1)
    model = build_lgbm().fit(x, StandardScaler().fit_transform(y).ravel())
    mean_abs_shap[target] = np.abs(shap.TreeExplainer(model).shap_values(x)).mean(axis=0)
mean_abs_shap = pd.DataFrame(mean_abs_shap, index=features)
spearman = model_input[features + targets].corr(method="spearman").loc[features, targets]

OUT_DIR.mkdir(parents=True, exist_ok=True)
mean_abs_shap.to_csv(OUT_DIR / "mean_abs_shap_covariates_plus_diet_baseline.csv")
spearman.to_csv(OUT_DIR / "spearman_feature_species_correlation_baseline.csv")
(mean_abs_shap * np.sign(spearman)).to_csv(OUT_DIR / "directional_mean_abs_shap_baseline.csv")
print(mean_abs_shap.mean(axis=1).sort_values(ascending=False).head(20))